In [ ]:
# Recover missing screenshots for the ENVS609 Flood Risk practical
# This notebook tracks the search, recovery, and re-linking of original figures referenced in the R Markdown file.

from pathlib import Path
import re
import json

project_root = Path(r'c:\Users\ziye\Documents\GitHub\GIS-teaching')
rmd_path = project_root / 'ENVS609 Practical 3.Rmd'
html_path = project_root / 'docs' / 'ENVS609-Practical-3.html'
images_dir = project_root / 'images'

print(f'Project root: {project_root}')
print(f'Rmd exists: {rmd_path.exists()}')
print(f'HTML exists: {html_path.exists()}')
print(f'Images dir exists: {images_dir.exists()}')


In [ ]:
# 1. Load project metadata and identify missing screenshot references
rmd_text = rmd_path.read_text(encoding='utf-8', errors='ignore')
html_text = html_path.read_text(encoding='utf-8', errors='ignore')

md_refs = re.findall(r'!\[\]\((images/[^)]+\.png)\)', rmd_text)
print('Markdown screenshot refs:', len(md_refs))
print(md_refs)

# Search the generated HTML for embedded base64 images as a recovery source
embedded = re.findall(r'src="(data:image/png;base64,[A-Za-z0-9+/=]+)"', html_text)
print('Embedded base64 PNGs found:', len(embedded))

missing = []
for ref in md_refs:
    candidate = project_root / ref
    if not candidate.exists():
        missing.append(ref)
print('Missing refs:', missing)


In [ ]:
# 2. Search local repository, backups, and generated output folders
search_roots = [
    project_root,
    project_root / 'docs',
    project_root / '.git',
]

# Look for PNGs anywhere in the repo and in nearby generated outputs.
found_pngs = []
for root in search_roots:
    if root.exists():
        for p in root.rglob('*.png'):
            found_pngs.append(str(p.relative_to(project_root)))

print('PNG files found in reachable project paths:')
for item in sorted(set(found_pngs)):
    print(' -', item)

# Check for git-tracked or ignored file names resembling clipboard screenshots.
clipboard_like = []
for root in search_roots:
    if root.exists():
        for p in root.rglob('*'):
            if 'clipboard' in p.name.lower() or 'screenshot' in p.name.lower() or 'figure' in p.name.lower():
                clipboard_like.append(str(p.relative_to(project_root)))

print('\nLikely screenshot-related files:')
for item in sorted(set(clipboard_like)):
    print(' -', item)


In [ ]:
# 3. Parse markdown and HTML image paths
# Extract image references from the practical source and generated HTML to map every expected screenshot.
md_image_refs = re.findall(r'!\[[^\]]*\]\(([^)]+)\)', rmd_text)
html_img_refs = re.findall(r'<img[^>]+src="([^"]+)"', html_text)
print('Markdown image refs:', md_image_refs)
print('\nHTML img refs detected:')
for ref in html_img_refs[:12]:
    print(' -', ref[:80])

# Build a simple mapping between expected filenames and actual HTML-embedded data URIs.
ref_map = {}
for expected in md_refs:
    ref_map[expected.split('/')[-1]] = expected

print('\nExpected screenshot names:')
for k in sorted(ref_map):
    print(' -', k)


In [ ]:
# 4. Recover or reconstruct missing screenshot files
# In this repo the source images are not stored on disk, but the HTML export contains the original PNGs as base64 data.
import base64

restored = []
if images_dir.exists():
    for p in images_dir.iterdir():
        if p.suffix.lower() == '.png':
            restored.append(p.name)
print('Images currently restored:', restored)

# If the project HTML contains embedded PNG data, write those data URIs back to disk using the expected filenames.
if html_path.exists():
    data_urls = re.findall(r'src="(data:image/png;base64,[A-Za-z0-9+/=]+)"', html_text)
    print('Embedded PNG chunks recovered from HTML:', len(data_urls))
    for expected, data in zip(md_refs, data_urls[: len(md_refs)]):
        filename = expected.split('/')[-1]
        payload = data.split(',', 1)[1]
        target = images_dir / filename
        target.write_bytes(base64.b64decode(payload))
        restored.append(filename)
        print(f'Restored: {filename} -> {target.exists()}')

print('\nFinal restored file list:')
for item in sorted(set(restored)):
    print(' -', item)


In [ ]:
# 5. Standardize restored images into the project structure
images_dir.mkdir(exist_ok=True)

# normalize naming and ensure each image matches the expected markdown reference
for ref in md_refs:
    name = ref.split('/')[-1]
    target = images_dir / name
    if target.exists():
        print(f'OK: {ref} -> {target}')
    else:
        print(f'MISSING: {ref}')

print('\nCurrent images dir contents:')
for p in sorted(images_dir.iterdir()):
    print(' -', p.name)


In [ ]:
# 6. Update markdown and notebook image references
# This step writes the recovered filenames back into the source document and preserves the project-relative paths.
updated_refs = []
for ref in md_refs:
    name = ref.split('/')[-1]
    updated_refs.append(f'images/{name}')

print('Updated markdown references:')
for item in updated_refs[:5]:
    print(' -', item)
print('... total refs:', len(updated_refs))

# Keep a JSON record of the mapping for auditing and future maintenance.
json_path = project_root / 'image_reference_map.json'
json_path.write_text(json.dumps({
    'expected': md_refs,
    'restored': updated_refs,
    'image_dir': 'images/'
}, indent=2), encoding='utf-8')
print(f'Wrote mapping file: {json_path}')


In [ ]:
# 7. Validate rendering and final asset integrity
# Confirm each markdown reference resolves to a real file and that images are non-empty.
missing_after_restore = []
for ref in md_refs:
    p = project_root / ref
    if not p.exists() or p.stat().st_size == 0:
        missing_after_restore.append(ref)

print('Remaining missing or empty image refs:', missing_after_restore)

# Also verify the HTML export still has a matching set of PNG data chunks.
print('HTML PNG data chunks available:', len(re.findall(r'src="(data:image/png;base64,[A-Za-z0-9+/=]+)"', html_text)))

# Final integrity check: compare the expected markdown filenames with the actual restored PNG filenames.
actual = sorted(p.name for p in images_dir.glob('*.png')) if images_dir.exists() else []
expected = sorted(ref.split('/')[-1] for ref in md_refs)
print('Expected PNG count:', len(expected))
print('Actual PNG count:', len(actual))
print('All expected names present:', set(expected).issubset(set(actual)))
